In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import torch
import re
import string
import math
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import os

# Konfigurasi Path
DATA_PATH = '../data/processed/tafsir_clean.csv'
MODEL_SBERT_PATH = '../models/sbert_tafsir_finetuned'
MODEL_XGB_PATH = '../models/xgboost_best_model.json'


# Fungsi Preprocessing

In [ ]:
# Inisialisasi Stopword Remover
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text) 
    text = stopword_remover.remove(text) 
    text = " ".join(text.split())
    return text

# Memuat Data dan Model

In [ ]:
# Memuat Korpus Tafsir
try:
    df_tafsir = pd.read_csv(DATA_PATH)
    print(f"Korpus Tafsir dimuat: {len(df_tafsir)} ayat.")
except FileNotFoundError:
    print("Error: File tafsir_clean.csv tidak ditemukan.")

# Membangun Index BM25
corpus_clean = df_tafsir['tafsir_text'].apply(clean_text).tolist()
tokenized_corpus = [doc.split() for doc in corpus_clean]
bm25 = BM25Okapi(tokenized_corpus)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sbert_model = SentenceTransformer(MODEL_SBERT_PATH, device=device)
corpus_embeddings = sbert_model.encode(df_tafsir['tafsir_text'].tolist(), convert_to_tensor=True, show_progress_bar=True)

bst = xgb.Booster()
try:
    bst.load_model(MODEL_XGB_PATH)
    print("Model XGBoost berhasil dimuat.")
except Exception as e:
    print(f"Gagal memuat model XGBoost: {e}")

# Definisi Data Uji

In [ ]:
# Data Uji Robustness
test_data = [
    # ID 1: Warisan
    {"id": "1a", "query": "Hukum warisan bagi perempuan", "target_surat": "An-Nisa'", "target_ayat": 11},
    {"id": "1b", "query": "Hukum pembagian harta setelah seseorang meninggal bagi perempuan", "target_surat": "An-Nisa'", "target_ayat": 11},
    
    # ID 2: Puasa
    {"id": "2a", "query": "perintah allah untuk kita melaksanakan puasa ramadhan", "target_surat": "Al-Baqarah", "target_ayat": 185},
    {"id": "2b", "query": "perintah allah untuk kita melaksanakan shaum di bulan suci", "target_surat": "Al-Baqarah", "target_ayat": 185},
    
    # ID 3: Sholat Jumat
    {"id": "3a", "query": "Cara melaksanakan sholat Jumat", "target_surat": "Al-Jumu'ah", "target_ayat": 9},
    {"id": "3b", "query": "Ketentuan melaksanakan sembahyang Jumat", "target_surat": "Al-Jumu'ah", "target_ayat": 9},

    # ID 4: Sumpah
    {"id": "4a", "query": "Denda bagi yang bersumpah palsu", "target_surat": "Al-Ma'idah", "target_ayat": 89},
    {"id": "4b", "query": "Konsekuensi sumpah dusta", "target_surat": "Al-Ma'idah", "target_ayat": 89},

    # ID 5: Riba
    {"id": "5a", "query": "Larangan memakan harta riba", "target_surat": "Ali 'Imran", "target_ayat": 130},
    {"id": "5b", "query": "Larangan memakan harta dari pinjaman yang berbunga", "target_surat": "Ali 'Imran", "target_ayat": 130},
    
    # ID 6: Khamr
    {"id": "6a", "query": "Apa itu khamar", "target_surat": "Al-Ma'idah", "target_ayat": 90},
    {"id": "6b", "query": "Definisi minuman memabukkan", "target_surat": "Al-Ma'idah", "target_ayat": 90},
]

print(f"Jumlah data uji: {len(test_data)}")

# Fungsi Prediksi Peringkat

In [ ]:
def predict_rank_system(query_text, target_surat, target_ayat, top_k=10):
    # Preprocessing Query
    clean_q = clean_text(query_text)
    q_tokens = clean_q.split()
    
    # Candidate Retrieval (SBERT)
    q_emb = sbert_model.encode(query_text, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, corpus_embeddings, top_k=50)[0]
    
    candidates = []
    
    # Feature Extraction
    for hit in hits:
        doc_idx = hit['corpus_id']
        clean_d = corpus_clean[doc_idx]
        d_tokens = clean_d.split()
        set_q, set_d = set(q_tokens), set(d_tokens)
        intersect = len(set_q & set_d)
        union = len(set_q | set_d)
        
        jaccard = intersect / union if union > 0 else 0
        overlap = intersect / len(set_q) if len(set_q) > 0 else 0
        bm25_score = bm25.get_batch_scores(q_tokens, [doc_idx])[0]
        
        candidates.append({
            'original_idx': doc_idx,
            'surat': df_tafsir.iloc[doc_idx]['surat'],
            'ayat': df_tafsir.iloc[doc_idx]['ayat'],
            'sbert_sim': hit['score'],
            'bm25_score': bm25_score,
            'jaccard_score': jaccard,
            'overlap_score': overlap
        })
        
    df_cand = pd.DataFrame(candidates)
    # Pastikan urutan fitur sama dengan saat training
    features = ['sbert_sim', 'bm25_score', 'jaccard_score', 'overlap_score']
    dmatrix = xgb.DMatrix(df_cand[features])
    df_cand['final_score'] = bst.predict(dmatrix)
    
    # Urutkan berdasarkan skor final
    df_cand = df_cand.sort_values('final_score', ascending=False).reset_index(drop=True)
    
    # Cek Posisi Target
    target_rank = -1
    
    # Normalisasi nama target untuk pencocokan
    tgt_surat_norm = str(target_surat).strip().lower().replace("'", "").replace("’", "")
    tgt_ayat_norm = int(target_ayat)
    
    for i, row in df_cand.iterrows():
        curr_surat_norm = str(row['surat']).strip().lower().replace("'", "").replace("’", "")
        curr_ayat_norm = int(row['ayat'])
        
        if curr_surat_norm == tgt_surat_norm and curr_ayat_norm == tgt_ayat_norm:
            target_rank = i + 1
            break
            
    # Kembalikan rank dan 5 hasil teratas untuk display
    top_5_results = []
    for i in range(min(5, len(df_cand))):
        top_5_results.append(f"{df_cand.iloc[i]['surat']} : {df_cand.iloc[i]['ayat']}")
        
    return target_rank, top_5_results

# Eksekusi Pengujian dan Perhitungan Metrik

In [ ]:
# Helper function untuk nDCG
def calculate_ndcg(rank, k=5):
    if rank == -1 or rank > k:
        return 0.0
    return 1.0 / math.log2(rank + 1)

results = []
total_mrr = 0
total_p_at_5 = 0
total_ndcg_at_5 = 0
k = 5

for item in test_data:
    rank, top_results = predict_rank_system(item['query'], item['target_surat'], item['target_ayat'])
    
    # Hitung Metrik per Kueri
    rr = 1.0 / rank if rank != -1 else 0.0
    p_5 = 1.0 if (rank != -1 and rank <= k) else 0.0
    ndcg_5 = calculate_ndcg(rank, k)
    
    # Akumulasi
    total_mrr += rr
    total_p_at_5 += p_5
    total_ndcg_at_5 += ndcg_5
    
    results.append({
        'ID': item['id'],
        'Query': item['query'],
        'Target': f"{item['target_surat']} : {item['target_ayat']}",
        'Rank Ditemukan': rank if rank != -1 else ">50",
        'Top 5 Prediksi': " | ".join(top_results),
        'P@5': p_5,
        'RR': rr,
        'nDCG@5': ndcg_5
    })

# Buat DataFrame Hasil
df_results = pd.DataFrame(results)

# Hitung Rata-Rata Metrik
avg_mrr = total_mrr / len(test_data)
avg_p_at_5 = total_p_at_5 / len(test_data)
avg_ndcg_at_5 = total_ndcg_at_5 / len(test_data)

print("\n" + "="*50)
print("HASIL AKHIR SKENARIO 2 (ROBUSTNESS TEST)")
print(f"Jumlah Kueri: {len(test_data)}")
print(f"Average Precision at 5 (P@5) : {avg_p_at_5:.2%}") 
print(f"Mean Reciprocal Rank (MRR)   : {avg_mrr:.4f}")    
print(f"nDCG at 5 (nDCG@5)           : {avg_ndcg_at_5:.4f}") 

# Tampilkan Tabel Detail
pd.set_option('display.max_colwidth', None)
display(df_results[['ID', 'Query', 'Target', 'Rank Ditemukan', 'Top 5 Prediksi']])
